# RAG Chatbot with Hugging Face + FAISS (GPU with Mistral 7B)

In [1]:
from huggingface_hub import login
login(token="your own")

In [2]:
from huggingface_hub import whoami
print(whoami())


{'type': 'user', 'id': '68918b4e55f301317256ff73', 'name': 'shashy404', 'fullname': 'Shashwati Buragohain', 'isPro': False, 'avatarUrl': '/avatars/b3a4556d2380aa51bc4bfcebde54fa6a.svg', 'orgs': [], 'auth': {'type': 'access_token', 'accessToken': {'displayName': 'image2speech', 'role': 'fineGrained', 'createdAt': '2025-08-05T04:46:55.238Z', 'fineGrained': {'canReadGatedRepos': True, 'global': [], 'scoped': [{'entity': {'_id': '65770c3426ef61bbf101d4da', 'type': 'model', 'name': 'mistralai/Mistral-7B-Instruct-v0.2'}, 'permissions': ['repo.content.read']}, {'entity': {'_id': '68918b4e55f301317256ff73', 'type': 'user', 'name': 'shashy404'}, 'permissions': ['repo.content.read']}]}}}}


In [3]:
from transformers import AutoTokenizer, AutoModelForCausalLM

model_name = "mistralai/Mistral-7B-Instruct-v0.2"

tokenizer = AutoTokenizer.from_pretrained(model_name)
model = AutoModelForCausalLM.from_pretrained(model_name)


Loading checkpoint shards:   0%|          | 0/3 [00:00<?, ?it/s]

In [4]:
from transformers import AutoModelForCausalLM, AutoTokenizer, pipeline

model_name = "mistralai/Mistral-7B-Instruct-v0.2"

tokenizer = AutoTokenizer.from_pretrained(model_name)
model = AutoModelForCausalLM.from_pretrained(
    model_name,
    device_map="auto",
    torch_dtype="auto"
)

rag_pipeline = pipeline(
    "text-generation",
    model=model,
    tokenizer=tokenizer,
    #device=0
)

Loading checkpoint shards:   0%|          | 0/3 [00:00<?, ?it/s]

Some parameters are on the meta device because they were offloaded to the cpu and disk.
Device set to use cpu


### Function to generate answers with context + sources

In [5]:
def generate_answer(query, retrieved_chunks):
    context = "\n".join([chunk['text'] for chunk in retrieved_chunks])
    prompt = f"""You are a helpful assistant.
Use the following context to answer the question.
If the answer is not in the context, say you don't know.

Context:
{context}

Question: {query}
Answer with explanation and include sources at the end in format: (source: filename)."""

    response = rag_pipeline(prompt, max_new_tokens=300, do_sample=True, temperature=0.3)
    return response[0]["generated_text"]

### Example Usage

In [6]:
# Example retrieved chunks (replace with your FAISS retrieval results)
retrieved_chunks = [
    {"text": "ADMM is a powerful optimization method that decomposes a problem into subproblems."},
    {"text": "It alternates between local variable updates and a global consensus step."}
]

query = "What is ADMM and how does it work?"
print(generate_answer(query, retrieved_chunks))

Setting `pad_token_id` to `eos_token_id`:2 for open-end generation.


You are a helpful assistant.
Use the following context to answer the question.
If the answer is not in the context, say you don't know.

Context:
ADMM is a powerful optimization method that decomposes a problem into subproblems.
It alternates between local variable updates and a global consensus step.

Question: What is ADMM and how does it work?
Answer with explanation and include sources at the end in format: (source: filename).

Answer:
ADMM stands for Alternating Direction Method of Multipliers. It is a powerful optimization method used to solve large-scale convex optimization problems. The method decomposes the problem into smaller subproblems and alternates between updating the local variables and performing a global consensus step (source: Boyd, S., & Vandenberghe, L. (2011). Distributed optimization and large-scale systems. Foundations and Trends® in Optimization: Vol. 2: No. 1-2, 1-100).

The basic idea behind ADMM is to split the original problem into two subproblems, each of